# NB11 — Building a Reading List and Research Map

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours  

---

## Motivation

A reading list that serves both Track A (India RA/JRF jobs in 6 months) and Track B (European PhD applications in 12 months) must be built deliberately, not accumulated by following whatever citation looks interesting. This notebook builds a priority-sorted `ReadingList` that manages papers across all 20 modules, tracks reading progress, and identifies where to invest the next 2 hours.

## 1. Paper Tier System

| Tier | Name | Standard | Examples |
|------|------|----------|----------|
| F | **Foundational** | Read all three passes; can reconstruct method unaided | Needleman-Wunsch, DESeq2, AlphaFold2 |
| L | **Landmark** | Read Pass 1 and Pass 2; know the main result and why it mattered | BLAST, Seurat, STAR |
| S | **Survey** | Read Pass 1 only; know it exists and what it claimed | Any paper in a Tier 3 module |
| U | **Unread** | On the list but not started | Everything in papers.md not yet read |

## 2. Priority Scoring Formula

Each paper gets a priority score:

$$\text{score} = (\text{TrackA} \times 2 + \text{TrackB} + \text{pass3\_gap\_penalty})$$

Where:
- `TrackA` ∈ [0, 5]: how important for India bioinformatics interviews
- `TrackB` ∈ [0, 5]: how important for European PhD applications
- `pass3_gap_penalty` ∈ {0, 3}: +3 if this paper is foundational (Tier F) but Pass 3 not yet done

**Logic:** Track A is weighted 2× because the 6-month deadline is harder. Pass-3 gap penalty prioritizes papers where you have read but not reconstructed — the highest-leverage action.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


@dataclass
class ReadingListEntry:
    title: str
    authors: str
    year: int
    module: str            # e.g. 'Module 08'
    tier: str              # 'F', 'L', 'S', 'U'
    track_a_relevance: float  # 0-5
    track_b_relevance: float  # 0-5
    pass1_done: bool = False
    pass2_done: bool = False
    pass3_success: Optional[bool] = None   # None = not attempted
    doi: str = ''

    def pass3_gap_penalty(self) -> float:
        """Return 3 if this is a Foundational paper without successful Pass 3."""
        if self.tier == 'F' and self.pass3_success is not True:
            return 3.0
        return 0.0

    def priority_score(self) -> float:
        return self.track_a_relevance * 2 + self.track_b_relevance + self.pass3_gap_penalty()

    def reading_status(self) -> str:
        if self.pass3_success is True:
            return 'Pass3-success'
        if self.pass3_success is False:
            return 'Pass3-failed'
        if self.pass2_done:
            return 'Pass2'
        if self.pass1_done:
            return 'Pass1'
        return 'Unread'

    def estimated_hours(self) -> float:
        """Approximate hours needed to complete to recommended depth."""
        if self.tier == 'F':
            remaining = {'Unread': 3.0, 'Pass1': 2.5, 'Pass2': 1.5,
                         'Pass3-failed': 1.0, 'Pass3-success': 0.0}
        elif self.tier == 'L':
            remaining = {'Unread': 1.5, 'Pass1': 1.0, 'Pass2': 0.0,
                         'Pass3-failed': 0.0, 'Pass3-success': 0.0}
        else:
            remaining = {'Unread': 0.15, 'Pass1': 0.0, 'Pass2': 0.0,
                         'Pass3-failed': 0.0, 'Pass3-success': 0.0}
        return remaining.get(self.reading_status(), 0.0)


class ReadingList:
    """Manage papers across all 20 modules with priority sorting."""

    def __init__(self):
        self.entries: List[ReadingListEntry] = []

    def add(self, entry: ReadingListEntry) -> None:
        self.entries.append(entry)

    def priority_sorted(self) -> List[ReadingListEntry]:
        return sorted(self.entries, key=lambda e: e.priority_score(), reverse=True)

    def coverage_by_module(self) -> Dict[str, Dict[str, int]]:
        """Count papers per module by reading status."""
        coverage = defaultdict(lambda: defaultdict(int))
        for e in self.entries:
            coverage[e.module][e.reading_status()] += 1
        return {mod: dict(statuses) for mod, statuses in coverage.items()}

    def total_hours_remaining(self) -> float:
        return sum(e.estimated_hours() for e in self.entries)

    def summary(self) -> dict:
        by_tier = defaultdict(int)
        by_status = defaultdict(int)
        for e in self.entries:
            by_tier[e.tier] += 1
            by_status[e.reading_status()] += 1
        return {
            'total': len(self.entries),
            'by_tier': dict(by_tier),
            'by_status': dict(by_status),
            'hours_remaining': round(self.total_hours_remaining(), 1),
        }


# Populate with papers from across the 20-module programme
rl = ReadingList()

programme_papers = [
    ReadingListEntry('Needleman-Wunsch alignment', 'Needleman & Wunsch', 1970, 'Module 08', 'F', 4.5, 3.0, True, True, True, '10.1016/0022-2836(70)90057-4'),
    ReadingListEntry('Smith-Waterman local alignment', 'Smith & Waterman', 1981, 'Module 08', 'F', 4.0, 2.5, True, True, False, '10.1016/0022-2836(81)90087-5'),
    ReadingListEntry('BLAST heuristic alignment', 'Altschul et al.', 1990, 'Module 08', 'F', 5.0, 4.0, True, False, None, '10.1016/S0022-2836(05)80360-2'),
    ReadingListEntry('STAR RNA-seq aligner', 'Dobin et al.', 2013, 'Module 09', 'L', 5.0, 3.5, True, True, None, '10.1093/bioinformatics/bts635'),
    ReadingListEntry('BWA-MEM aligner', 'Li', 2013, 'Module 09', 'L', 4.5, 3.0, True, False, None, 'arXiv:1303.3997'),
    ReadingListEntry('DESeq2', 'Love et al.', 2014, 'Module 10', 'F', 5.0, 4.5, True, True, True, '10.1186/s13059-014-0550-8'),
    ReadingListEntry('Seurat single-cell integration', 'Butler et al.', 2018, 'Module 10', 'L', 4.0, 4.0, False, False, None, '10.1038/nbt.4096'),
    ReadingListEntry('SCANPY', 'Wolf et al.', 2018, 'Module 10', 'L', 4.0, 4.0, True, False, None, '10.1186/s13059-017-1382-0'),
    ReadingListEntry('Harmony integration', 'Korsunsky et al.', 2019, 'Module 10', 'L', 3.5, 3.5, False, False, None, '10.1038/s41592-019-0619-0'),
    ReadingListEntry('UMAP', 'McInnes et al.', 2018, 'Module 13', 'L', 3.5, 4.0, True, False, None, 'arXiv:1802.03426'),
    ReadingListEntry('AlphaFold2', 'Jumper et al.', 2021, 'Module 11', 'F', 3.5, 5.0, False, False, None, '10.1038/s41586-021-03819-2'),
    ReadingListEntry('Protein language models', 'Lin et al.', 2022, 'Module 11', 'L', 2.5, 4.5, False, False, None, '10.1126/science.ade2574'),
    ReadingListEntry('Why most findings are false', 'Ioannidis', 2005, 'Module 19', 'F', 2.0, 2.5, True, True, False, '10.1371/journal.pmed.0020124'),
    ReadingListEntry('How to Read a Paper', 'Keshav', 2007, 'Module 19', 'F', 2.5, 2.0, True, True, True, '10.1145/1273445.1273458'),
    ReadingListEntry('Benchmarking omics tools', 'Mangul et al.', 2019, 'Module 19', 'L', 3.0, 3.0, False, False, None, '10.1038/s41467-019-09406-4'),
    ReadingListEntry('Ten rules literature review', 'Pautasso', 2013, 'Module 19', 'S', 1.5, 2.0, False, False, None, '10.1371/journal.pcbi.1003149'),
    ReadingListEntry('scRNA-seq best practices', 'Luecken & Theis', 2019, 'Module 10', 'L', 4.0, 4.5, False, False, None, '10.15252/msb.20188746'),
    ReadingListEntry('edgeR for count data', 'Robinson et al.', 2010, 'Module 10', 'L', 4.0, 3.5, True, False, None, '10.1093/bioinformatics/btp616'),
    ReadingListEntry('Random forests', 'Breiman', 2001, 'Module 13', 'L', 3.5, 3.0, True, True, None, '10.1023/A:1010933404324'),
    ReadingListEntry('Attention is all you need', 'Vaswani et al.', 2017, 'Module 14', 'L', 3.0, 4.5, False, False, None, 'arXiv:1706.03762'),
]

for paper in programme_papers:
    rl.add(paper)

s = rl.summary()
print(f"Reading list summary: {s['total']} papers, {s['hours_remaining']} hours remaining")
print(f"By tier: {s['by_tier']}")
print(f"By status: {s['by_status']}")
print("\nTop 5 priority papers:")
for i, e in enumerate(rl.priority_sorted()[:5]):
    print(f"  {i+1}. [{e.module}] {e.authors} ({e.year}) — score={e.priority_score():.1f}, status={e.reading_status()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Reading List Analysis — Coverage and Priority', fontsize=12, fontweight='bold')

# Panel 1: Reading coverage heatmap across modules
ax1 = axes[0]
modules = sorted(set(e.module for e in rl.entries))
statuses = ['Unread', 'Pass1', 'Pass2', 'Pass3-failed', 'Pass3-success']
cov = rl.coverage_by_module()
heatmap_data = np.array([[cov.get(m, {}).get(s, 0) for s in statuses] for m in modules])

im = ax1.imshow(heatmap_data, cmap='Blues', aspect='auto')
ax1.set_xticks(range(len(statuses)))
ax1.set_xticklabels([s.replace('-', '\n') for s in statuses], fontsize=8, rotation=30)
ax1.set_yticks(range(len(modules)))
ax1.set_yticklabels([m.replace('Module ', 'M') for m in modules], fontsize=8)
ax1.set_title('Reading Coverage Heatmap\n(papers per module × status)')
plt.colorbar(im, ax=ax1, label='Papers')

for i in range(len(modules)):
    for j in range(len(statuses)):
        if heatmap_data[i, j] > 0:
            ax1.text(j, i, int(heatmap_data[i, j]), ha='center', va='center',
                     fontsize=8, color='white' if heatmap_data[i, j] > 2 else 'black')

# Panel 2: Priority queue bar chart (top 10)
ax2 = axes[1]
top10 = rl.priority_sorted()[:10]
labels10 = [f"{e.authors.split()[0]} ({e.year})" for e in top10]
scores10 = [e.priority_score() for e in top10]
status_colors = {
    'Unread': '#E53935',
    'Pass1': '#FB8C00',
    'Pass2': '#FDD835',
    'Pass3-failed': '#43A047',
    'Pass3-success': '#1E88E5',
}
bar_colors10 = [status_colors.get(e.reading_status(), 'gray') for e in top10]
y_pos = np.arange(len(labels10))
ax2.barh(y_pos, scores10, color=bar_colors10, alpha=0.85)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(labels10, fontsize=9)
ax2.set_xlabel('Priority score')
ax2.set_title('Top 10 Priority Papers\n(color = reading status)')
ax2.grid(True, axis='x', alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in status_colors.items()]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=7)

# Panel 3: Reading time estimate
ax3 = axes[2]
tier_hours = defaultdict(float)
tier_counts = defaultdict(int)
for e in rl.entries:
    tier_hours[e.tier] += e.estimated_hours()
    tier_counts[e.tier] += 1

tiers = ['F', 'L', 'S']
tier_labels = ['Foundational\n(3 hr each)', 'Landmark\n(1.5 hr each)', 'Survey\n(10 min each)']
hours_val = [tier_hours.get(t, 0) for t in tiers]
counts_val = [tier_counts.get(t, 0) for t in tiers]
colors_tier = ['#E53935', '#FB8C00', '#43A047']
ax3.bar(tier_labels, hours_val, color=colors_tier, alpha=0.85)
for i, (h, c) in enumerate(zip(hours_val, counts_val)):
    ax3.text(i, h + 0.1, f'{h:.1f} hr\n({c} papers)', ha='center', va='bottom', fontsize=9)
ax3.set_ylabel('Hours remaining')
ax3.set_title(f'Reading Time by Tier\n(Total: {rl.total_hours_remaining():.1f} hrs remaining)')
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('reading_list_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Add 5 papers from your current active module to the ReadingList. Assign correct tier, Track A and Track B relevance scores. Where do they appear in the priority ranking?

**Exercise 2:** Identify the single highest-priority unread foundational paper in your ReadingList. Commit to reading it this week (Pass 1 + Pass 2 minimum).

**Exercise 3:** Compute how many hours remain to achieve Pass-2 or better for all Foundational papers. Is this achievable in the next 4 months (assuming 2 hours/week)?

**Exercise 4 (reflection):** Why does the priority formula weight Track A at 2× relative to Track B? Under what circumstances would you change this weighting?

## Quiz

1. What are the four tiers in the paper tier system?
2. What triggers the pass3_gap_penalty?
3. Why is BLAST scored higher on Track A relevance than AlphaFold2?
4. What is the maximum possible priority score, and what paper characteristics produce it?
5. If you had 4 hours this week, which two papers from the top-10 list above should you prioritize and why?

## References

- All papers in this module's papers.md
- Connected Papers (https://www.connectedpapers.com) for finding related works